In [1]:
!pip install linearmodels -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 16.9 MB/s eta 0:00:00


In [2]:
# ══════════════════════════════════════════════════════════════
# UPTOWN ANALYSTS — Full Restart (Raw Hourly, No Aggregation)
# ══════════════════════════════════════════════════════════════

import os, re, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted\n")


Mounted at /content/drive
✅ Drive mounted



In [3]:
# ── PATHS ─────────────────────────────────────────────────────
PATH_2024     = '/content/drive/MyDrive/MTA/clean_data/mta_2024_cleaned_cj.csv'
PATH_2025     = '/content/drive/MyDrive/MTA/clean_data/MTA_Daily_Clean_2025.csv'
PATH_STATIONS = '/content/drive/MyDrive/MTA/MTA Map/MTA_Subway_Stations_20260402.csv'
PATH_VEHICLES = '/content/drive/MyDrive/MTA/CRZ/MTA_Congestion_Relief_Zone_Vehicle_Entries__Beginning_2025_20260402.csv'
PATH_EVASION  = '/content/drive/MyDrive/MTA/Fare Evasion/MTA_NYCT_Subway_Fare_Evasion__Beginning_2018_20260403.csv'

CHUNKSIZE = 500_000
CP_DATE   = pd.Timestamp('2025-01-05')

In [5]:
# ── CHUNKED LOADER — NO AGGREGATION ──────────────────────────
def load_raw(path, year_label):
    print(f"Loading {year_label} in chunks...")
    chunks = []
    total_rows = 0

    for i, chunk in enumerate(pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False)):
        total_rows += len(chunk)
        chunk['ridership'] = pd.to_numeric(chunk['ridership'], errors='coerce').fillna(0)
        chunk['transfers'] = pd.to_numeric(chunk['transfers'], errors='coerce').fillna(0)
        chunk['Date']      = pd.to_datetime(chunk['Date'], errors='coerce')
        chunk['Hour']      = pd.to_numeric(chunk['Hour'], errors='coerce').fillna(0).astype(int)
        chunk['year']      = year_label
        chunks.append(chunk)

        if (i + 1) % 10 == 0:
            print(f"  ...{total_rows:,} rows processed")

    combined = pd.concat(chunks, ignore_index=True)
    print(f"  ✅ {year_label}: {total_rows:,} rows loaded\n")
    return combined

raw_2024 = load_raw(PATH_2024, 2024)
raw_2025 = load_raw(PATH_2025, 2025)

Loading 2024 in chunks...
  ...5,000,000 rows processed
  ...10,000,000 rows processed
  ...15,000,000 rows processed
  ...20,000,000 rows processed
  ...25,000,000 rows processed
  ✅ 2024: 27,012,513 rows loaded

Loading 2025 in chunks...
  ...5,000,000 rows processed
  ...10,000,000 rows processed
  ...15,000,000 rows processed
  ...20,000,000 rows processed
  ...25,000,000 rows processed
  ...30,000,000 rows processed
  ...35,000,000 rows processed
  ✅ 2025: 35,382,811 rows loaded



In [6]:
# ── STACK ─────────────────────────────────────────────────────
df = pd.concat([raw_2024, raw_2025], ignore_index=True)
df = df.sort_values(['station_complex_id', 'Date', 'Hour']).reset_index(drop=True)
print(f"Master panel: {len(df):,} raw rows")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")

Master panel: 62,395,324 raw rows
Memory: 29617.0 MB



In [7]:
# ── JOIN STATION METADATA ─────────────────────────────────────
print("Joining MTA station metadata...")
stations = pd.read_csv(PATH_STATIONS)
stations = stations[['Complex ID', 'CBD', 'Division', 'Line', 'Daytime Routes', 'Structure']].copy()
stations = stations.rename(columns={
    'Complex ID'    : 'station_complex_id',
    'CBD'           : 'CBD_official',
    'Daytime Routes': 'Daytime_Routes'
})
stations['station_complex_id'] = stations['station_complex_id'].astype(str)
stations = stations.drop_duplicates('station_complex_id')

print(f"  Stations loaded: {len(stations)}")
print(f"  CBD column values: {stations['CBD_official'].unique()}")

df['station_complex_id'] = df['station_complex_id'].astype(str)
df = df.merge(stations, on='station_complex_id', how='left')

df['CBD_official'] = df['CBD_official'].map(
    {True: True, False: False, 'True': True, 'False': False}
).fillna(False)
df['treated'] = df['CBD_official'].astype(int)

print(f"  Treatment stations: {df[df['treated']==1]['station_complex_id'].nunique()}")
print(f"  Control stations  : {df[df['treated']==0]['station_complex_id'].nunique()}\n")


Joining MTA station metadata...
  Stations loaded: 445
  CBD column values: [False  True]
  Treatment stations: 65
  Control stations  : 363



In [8]:
# ── JOIN CRZ VEHICLE ENTRIES ──────────────────────────────────
print("Joining CRZ vehicle entries...")
vehicles = pd.read_csv(PATH_VEHICLES)
vehicles['Toll Date'] = pd.to_datetime(vehicles['Toll Date'], errors='coerce')
daily_vehicles = (vehicles.groupby('Toll Date')['CRZ Entries']
                           .sum().reset_index()
                           .rename(columns={'Toll Date': 'Date',
                                            'CRZ Entries': 'daily_crz_entries'}))
df = df.merge(daily_vehicles, on='Date', how='left')
df['daily_crz_entries'] = df['daily_crz_entries'].fillna(0)
print(f"  Vehicle dates matched: {df['daily_crz_entries'].gt(0).sum():,} rows\n")


Joining CRZ vehicle entries...
  Vehicle dates matched: 34,523,007 rows



In [9]:
# ── JOIN FARE EVASION ─────────────────────────────────────────
print("Joining fare evasion...")
evasion = pd.read_csv(PATH_EVASION)
evasion.columns = evasion.columns.str.strip()
evasion['period_start'] = pd.to_datetime(
    evasion['Time Period'].str.replace(
        r'(\d{4})-Q(\d)',
        lambda m: f"{m.group(1)}-{['01','04','07','10'][int(m.group(2))-1]}-01",
        regex=True), errors='coerce')
evasion['fare_evasion_pct'] = pd.to_numeric(
    evasion['Fare Evasion'].str.replace('%', '', regex=False), errors='coerce')
evasion = evasion[['period_start', 'fare_evasion_pct']].dropna()

df['quarter_start'] = df['Date'].dt.to_period('Q').dt.to_timestamp()
df = df.merge(evasion.rename(columns={'period_start': 'quarter_start'}),
              on='quarter_start', how='left')

Joining fare evasion...


In [10]:
# ── FIX FARE EVASION NULLS ────────────────────────────────────
df['fare_evasion_pct'] = (
    df.sort_values('Date')
      .groupby('station_complex_id')['fare_evasion_pct']
      .transform(lambda x: x.ffill().bfill())
)
global_evasion = df.groupby('quarter_start')['fare_evasion_pct'].transform('median')
df['fare_evasion_pct'] = df['fare_evasion_pct'].fillna(global_evasion)
print(f"  Evasion range: {df['fare_evasion_pct'].min():.1f}% – {df['fare_evasion_pct'].max():.1f}%\n")

  Evasion range: 9.8% – 14.0%



In [11]:
# ── DiD COLUMNS ───────────────────────────────────────────────
df['post']     = (df['Date'] >= CP_DATE).astype(int)
df['did_term'] = df['post'] * df['treated']

# ── TIME OF DAY BUCKETS ───────────────────────────────────────
def time_bucket(h):
    if 7 <= h <= 9:     return 'AM_peak'
    elif 17 <= h <= 19: return 'PM_peak'
    elif 10 <= h <= 16: return 'midday'
    elif 20 <= h <= 23: return 'evening'
    else:               return 'overnight'

df['time_of_day'] = df['Hour'].apply(time_bucket)

print("══ PANEL READY ══")
print(f"Shape  : {df.shape}")
print(f"Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Dates  : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"\nNull check:")
print(df[['treated','ridership','daily_crz_entries','fare_evasion_pct']].isna().sum())

══ PANEL READY ══
Shape  : (62395324, 26)
Memory : 50430.1 MB
Dates  : 2024-01-01 → 2026-03-25

Null check:
treated              0
ridership            0
daily_crz_entries    0
fare_evasion_pct     0
dtype: int64


In [12]:
# ══════════════════════════════════════════════════════════════
# HEAD & TAIL — 90 rows each
# ══════════════════════════════════════════════════════════════

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

print("── HEAD 90 ──────────────────────────────────────────────")
print(df.head(90).to_string())

print("\n── TAIL 90 ──────────────────────────────────────────────")
print(df.tail(90).to_string())

── HEAD 90 ──────────────────────────────────────────────
   station_complex_id             station_complex borough  latitude  longitude       Date Day_of_Week transit_timestamp               fare_class_category  ridership  transfers        CRZ_Zone  Hour  year  CBD_official Division     Line Daytime_Routes Structure  treated  daily_crz_entries quarter_start  fare_evasion_pct  post  did_term time_of_day
0                   1  Astoria-Ditmars Blvd (N,W)  Queens     40.78     -73.91 2025-01-01   Wednesday          00:00:00             Metrocard - Full Fare          7          0  Outer Boroughs     0  2025         False      BMT  Astoria            N W  Elevated        0               0.00    2025-01-01              9.80     0         0   overnight
1                   1  Astoria-Ditmars Blvd (N,W)  Queens     40.78     -73.91 2025-01-01   Wednesday          00:00:00                 Metrocard - Other          1          0  Outer Boroughs     0  2025         False      BMT  Astoria         

In [13]:
# ── DROP transit_timestamp ────────────────────────────────────
df = df.drop(columns=['transit_timestamp'])

print(f"Columns remaining : {len(df.columns)}")
print(f"Columns           : {df.columns.tolist()}")

Columns remaining : 25
Columns           : ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour', 'year', 'CBD_official', 'Division', 'Line', 'Daytime_Routes', 'Structure', 'treated', 'daily_crz_entries', 'quarter_start', 'fare_evasion_pct', 'post', 'did_term', 'time_of_day']


In [14]:
# ══════════════════════════════════════════════════════════════
# SANITY CHECK — Before Export
# ══════════════════════════════════════════════════════════════

print("═" * 60)
print("SANITY CHECK REPORT")
print("═" * 60)

# ── 1. SHAPE ──────────────────────────────────────────────────
print(f"\n── 1. Shape ──────────────────────────────────────────")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"Columns : {df.columns.tolist()}")

# ── 2. YEAR COVERAGE ──────────────────────────────────────────
print(f"\n── 2. Year coverage ──────────────────────────────────")
year_counts = df.groupby('year').size().rename('rows')
print(year_counts)
print(f"Date range 2024: {df[df['year']==2024]['Date'].min().date()} → {df[df['year']==2024]['Date'].max().date()}")
print(f"Date range 2025: {df[df['year']==2025]['Date'].min().date()} → {df[df['year']==2025]['Date'].max().date()}")

# ── 3. NULL CHECK — ALL COLUMNS ───────────────────────────────
print(f"\n── 3. Null check (all columns) ───────────────────────")
null_counts = df.isna().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("✅ No nulls anywhere in the dataset")
else:
    print("⚠️  Nulls found:")
    print(null_counts)

# ── 4. BLANK STRING CHECK ─────────────────────────────────────
print(f"\n── 4. Blank string check ─────────────────────────────")
str_cols = df.select_dtypes(include='object').columns
blank_counts = {col: (df[col].str.strip() == '').sum() for col in str_cols}
blank_counts = {k: v for k, v in blank_counts.items() if v > 0}
if len(blank_counts) == 0:
    print("✅ No blank strings in any column")
else:
    print("⚠️  Blank strings found:")
    print(blank_counts)

# ── 5. KEY COLUMN VALUE RANGES ────────────────────────────────
print(f"\n── 5. Key column value ranges ────────────────────────")
print(f"Hour            : {df['Hour'].min()} – {df['Hour'].max()} (expect 0–23)")
print(f"ridership       : {df['ridership'].min():,.0f} – {df['ridership'].max():,.0f}")
print(f"transfers       : {df['transfers'].min():,.0f} – {df['transfers'].max():,.0f}")
print(f"treated         : {sorted(df['treated'].unique())} (expect [0, 1])")
print(f"post            : {sorted(df['post'].unique())} (expect [0, 1])")
print(f"did_term        : {sorted(df['did_term'].unique())} (expect [0, 1])")
print(f"fare_evasion_pct: {df['fare_evasion_pct'].min():.1f}% – {df['fare_evasion_pct'].max():.1f}%")
print(f"daily_crz_entries 2024: {df[df['year']==2024]['daily_crz_entries'].max():,.0f} (expect 0)")
print(f"daily_crz_entries 2025: {df[df['year']==2025]['daily_crz_entries'].max():,.0f} (expect >0)")

# ── 6. STATION COVERAGE ───────────────────────────────────────
print(f"\n── 6. Station coverage ───────────────────────────────")
print(f"Total unique stations     : {df['station_complex_id'].nunique()}")
print(f"Treatment stations (CBD)  : {df[df['treated']==1]['station_complex_id'].nunique()}")
print(f"Control stations          : {df[df['treated']==0]['station_complex_id'].nunique()}")
stations_2024 = df[df['year']==2024]['station_complex_id'].nunique()
stations_2025 = df[df['year']==2025]['station_complex_id'].nunique()
print(f"Stations in 2024          : {stations_2024}")
print(f"Stations in 2025          : {stations_2025}")
if stations_2024 != stations_2025:
    print(f"⚠️  Station count mismatch between years — investigate before DiD")
else:
    print(f"✅ Same stations across both years")

# ── 7. POST FLAG SANITY ───────────────────────────────────────
print(f"\n── 7. Post flag sanity ───────────────────────────────")
print(df.groupby(['year','post']).size().rename('rows'))
pre_2025 = df[(df['year']==2025) & (df['post']==0)]['Date'].unique()
print(f"Pre-treatment days in 2025: {sorted(pre_2025)}")

# ── 8. TIME OF DAY DISTRIBUTION ───────────────────────────────
print(f"\n── 8. Time of day distribution ───────────────────────")
print(df['time_of_day'].value_counts())

# ── 9. DUPLICATE CHECK ────────────────────────────────────────
print(f"\n── 9. Duplicate check ────────────────────────────────")
key_cols = ['station_complex_id', 'Date', 'Hour', 'fare_class_category']
dupes = df.duplicated(subset=key_cols).sum()
if dupes == 0:
    print(f"✅ No duplicates on {key_cols}")
else:
    print(f"⚠️  {dupes:,} duplicate rows found on {key_cols}")

# ── FINAL VERDICT ─────────────────────────────────────────────
print(f"\n═" * 60)
issues = []
if len(null_counts) > 0:       issues.append("nulls present")
if len(blank_counts) > 0:      issues.append("blank strings present")
if dupes > 0:                  issues.append("duplicates present")
if stations_2024 != stations_2025: issues.append("station count mismatch")

if len(issues) == 0:
    print("✅ ALL CHECKS PASSED — safe to export")
else:
    print(f"⚠️  ISSUES FOUND: {', '.join(issues)}")
    print("Resolve before exporting")
print("═" * 60)

════════════════════════════════════════════════════════════
SANITY CHECK REPORT
════════════════════════════════════════════════════════════

── 1. Shape ──────────────────────────────────────────
Rows    : 62,395,324
Columns : 25
Columns : ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour', 'year', 'CBD_official', 'Division', 'Line', 'Daytime_Routes', 'Structure', 'treated', 'daily_crz_entries', 'quarter_start', 'fare_evasion_pct', 'post', 'did_term', 'time_of_day']

── 2. Year coverage ──────────────────────────────────
year
2024    27012513
2025    35382811
Name: rows, dtype: int64
Date range 2024: 2024-01-01 → 2024-12-31
Date range 2025: 2025-01-01 → 2026-03-25

── 3. Null check (all columns) ───────────────────────
⚠️  Nulls found:
Division          238696
Line              238696
Daytime_Routes    238696
Structure         238696
dtype: int64

── 4. Blank string c

In [15]:
# ── IDENTIFY NULL STATIONS ────────────────────────────────────
null_stations = (df[df['Division'].isna()]['station_complex_id']
                 .value_counts()
                 .reset_index()
                 .rename(columns={'index':'station_complex_id',
                                  'station_complex_id':'null_rows'}))
print(f"Stations with null metadata: {len(null_stations)}")
print(null_stations.to_string())

Stations with null metadata: 2
  null_rows   count
0     TRAM1  120905
1     TRAM2  117791


In [16]:
# ══════════════════════════════════════════════════════════════
# FIX — Tram metadata + drop 2026 + export
# ══════════════════════════════════════════════════════════════

# ── 1. HARDCODE TRAM METADATA ─────────────────────────────────
tram_meta = {
    'TRAM1': {
        'Division'      : 'Tram',
        'Line'          : 'Roosevelt Island Tramway',
        'Daytime_Routes': 'Tram',
        'Structure'     : 'Aerial'
    },
    'TRAM2': {
        'Division'      : 'Tram',
        'Line'          : 'Roosevelt Island Tramway',
        'Daytime_Routes': 'Tram',
        'Structure'     : 'Aerial'
    }
}

for tram_id, meta in tram_meta.items():
    mask = df['station_complex_id'] == tram_id
    for col, val in meta.items():
        df.loc[mask, col] = val

print("✅ Tram metadata imputed")
print(df[df['station_complex_id'].isin(['TRAM1','TRAM2'])][
    ['station_complex_id','station_complex','Division','Line',
     'Daytime_Routes','Structure']].drop_duplicates().to_string())

# ── 2. DROP 2026 ROWS ─────────────────────────────────────────
before = len(df)
df = df[df['Date'] <= pd.Timestamp('2025-12-31')].reset_index(drop=True)
after  = len(df)
print(f"\n✅ Dropped 2026 rows: {before - after:,} removed")
print(f"   Remaining rows   : {after:,}")
print(f"   Date range       : {df['Date'].min().date()} → {df['Date'].max().date()}")

# ── 3. FINAL NULL CHECK ───────────────────────────────────────
print(f"\nFinal null check:")
null_counts = df.isna().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("✅ No nulls anywhere")
else:
    print("⚠️  Nulls remaining:")
    print(null_counts)

# ── 4. YEAR COVERAGE CONFIRM ──────────────────────────────────
print(f"\nYear coverage:")
print(df.groupby('year').size().rename('rows'))
print(f"Date range 2024: {df[df['year']==2024]['Date'].min().date()} → {df[df['year']==2024]['Date'].max().date()}")
print(f"Date range 2025: {df[df['year']==2025]['Date'].min().date()} → {df[df['year']==2025]['Date'].max().date()}")

✅ Tram metadata imputed
         station_complex_id         station_complex Division                      Line Daytime_Routes Structure
62156628              TRAM1  RI Tramway (Manhattan)     Tram  Roosevelt Island Tramway           Tram    Aerial
62277533              TRAM2  RI Tramway (Roosevelt)     Tram  Roosevelt Island Tramway           Tram    Aerial

✅ Dropped 2026 rows: 4,911,039 removed
   Remaining rows   : 57,484,285
   Date range       : 2024-01-01 → 2025-12-31

Final null check:
✅ No nulls anywhere

Year coverage:
year
2024    27012513
2025    30471772
Name: rows, dtype: int64
Date range 2024: 2024-01-01 → 2024-12-31
Date range 2025: 2025-01-01 → 2025-12-31


In [17]:
# ══════════════════════════════════════════════════════════════
# FINAL SANITY CHECK — Post Tram Fix & 2026 Drop
# ══════════════════════════════════════════════════════════════

print("═" * 60)
print("FINAL SANITY CHECK REPORT")
print("═" * 60)

# ── 1. SHAPE ──────────────────────────────────────────────────
print(f"\n── 1. Shape ──────────────────────────────────────────")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"Columns : {df.columns.tolist()}")

# ── 2. YEAR & DATE COVERAGE ───────────────────────────────────
print(f"\n── 2. Year & date coverage ───────────────────────────")
print(df.groupby('year').size().rename('rows'))
print(f"\nDate range 2024: {df[df['year']==2024]['Date'].min().date()} → {df[df['year']==2024]['Date'].max().date()} (expect 2024-01-01 → 2024-12-31)")
print(f"Date range 2025: {df[df['year']==2025]['Date'].min().date()} → {df[df['year']==2025]['Date'].max().date()} (expect 2025-01-01 → 2025-12-31)")
print(f"\n2026 rows remaining: {(df['Date'] > pd.Timestamp('2025-12-31')).sum()} (expect 0)")

# ── 3. NULL CHECK — ALL COLUMNS ───────────────────────────────
print(f"\n── 3. Null check (all columns) ───────────────────────")
null_counts = df.isna().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("✅ No nulls anywhere in the dataset")
else:
    print("⚠️  Nulls found:")
    print(null_counts)

# ── 4. BLANK STRING CHECK ─────────────────────────────────────
print(f"\n── 4. Blank string check ─────────────────────────────")
str_cols = df.select_dtypes(include='object').columns
blank_counts = {col: (df[col].str.strip() == '').sum() for col in str_cols}
blank_counts = {k: v for k, v in blank_counts.items() if v > 0}
if len(blank_counts) == 0:
    print("✅ No blank strings in any column")
else:
    print("⚠️  Blank strings found:")
    print(blank_counts)

# ── 5. TRAM STATIONS CHECK ────────────────────────────────────
print(f"\n── 5. Tram stations metadata check ───────────────────")
tram_check = (df[df['station_complex_id'].isin(['TRAM1','TRAM2'])]
              [['station_complex_id','station_complex','Division',
                'Line','Daytime_Routes','Structure']]
              .drop_duplicates())
if tram_check[['Division','Line','Daytime_Routes','Structure']].isna().any().any():
    print("⚠️  Tram metadata still has nulls")
else:
    print("✅ Tram metadata clean")
print(tram_check.to_string())

# ── 6. KEY COLUMN VALUE RANGES ────────────────────────────────
print(f"\n── 6. Key column value ranges ────────────────────────")
print(f"Hour              : {df['Hour'].min()} – {df['Hour'].max()} (expect 0–23)")
print(f"ridership         : {df['ridership'].min():,.0f} – {df['ridership'].max():,.0f}")
print(f"transfers         : {df['transfers'].min():,.0f} – {df['transfers'].max():,.0f}")
print(f"treated           : {sorted(df['treated'].unique())} (expect [0, 1])")
print(f"post              : {sorted(df['post'].unique())} (expect [0, 1])")
print(f"did_term          : {sorted(df['did_term'].unique())} (expect [0, 1])")
print(f"fare_evasion_pct  : {df['fare_evasion_pct'].min():.1f}% – {df['fare_evasion_pct'].max():.1f}%")
print(f"daily_crz_entries 2024: {df[df['year']==2024]['daily_crz_entries'].max():,.0f} (expect 0)")
print(f"daily_crz_entries 2025: {df[df['year']==2025]['daily_crz_entries'].max():,.0f} (expect >0)")

# ── 7. STATION COVERAGE ───────────────────────────────────────
print(f"\n── 7. Station coverage ───────────────────────────────")
print(f"Total unique stations    : {df['station_complex_id'].nunique()}")
print(f"Treatment stations (CBD) : {df[df['treated']==1]['station_complex_id'].nunique()}")
print(f"Control stations         : {df[df['treated']==0]['station_complex_id'].nunique()}")
stations_2024 = df[df['year']==2024]['station_complex_id'].nunique()
stations_2025 = df[df['year']==2025]['station_complex_id'].nunique()
print(f"Stations in 2024         : {stations_2024}")
print(f"Stations in 2025         : {stations_2025}")
if stations_2024 != stations_2025:
    print(f"⚠️  Station count mismatch between years")
else:
    print(f"✅ Same stations across both years")

# ── 8. POST FLAG SANITY ───────────────────────────────────────
print(f"\n── 8. Post flag sanity ───────────────────────────────")
print(df.groupby(['year','post']).size().rename('rows'))
pre_2025 = df[(df['year']==2025) & (df['post']==0)]['Date'].unique()
print(f"Pre-treatment days in 2025: {sorted(pre_2025)}")

# ── 9. TIME OF DAY DISTRIBUTION ───────────────────────────────
print(f"\n── 9. Time of day distribution ───────────────────────")
print(df['time_of_day'].value_counts())

# ── 10. FARE CLASS DISTRIBUTION ───────────────────────────────
print(f"\n── 10. Fare class distribution ───────────────────────")
print(df['fare_class_category'].value_counts())

# ── 11. BOROUGH DISTRIBUTION ──────────────────────────────────
print(f"\n── 11. Borough distribution ──────────────────────────")
print(df.groupby('borough').size().rename('rows').sort_values(ascending=False))

# ── 12. DUPLICATE CHECK ───────────────────────────────────────
print(f"\n── 12. Duplicate check ───────────────────────────────")
key_cols = ['station_complex_id', 'Date', 'Hour', 'fare_class_category']
dupes = df.duplicated(subset=key_cols).sum()
if dupes == 0:
    print(f"✅ No duplicates on {key_cols}")
else:
    print(f"⚠️  {dupes:,} duplicate rows found on {key_cols}")

# ── FINAL VERDICT ─────────────────────────────────────────────
print(f"\n{'═' * 60}")
issues = []
if len(null_counts) > 0:                            issues.append("nulls present")
if len(blank_counts) > 0:                           issues.append("blank strings present")
if dupes > 0:                                       issues.append("duplicates present")
if stations_2024 != stations_2025:                  issues.append("station count mismatch")
if (df['Date'] > pd.Timestamp('2025-12-31')).sum(): issues.append("2026 rows still present")

if len(issues) == 0:
    print("✅ ALL CHECKS PASSED — safe to proceed to size estimate")
else:
    print(f"⚠️  ISSUES FOUND: {', '.join(issues)}")
    print("Resolve before proceeding")
print(f"{'═' * 60}")

════════════════════════════════════════════════════════════
FINAL SANITY CHECK REPORT
════════════════════════════════════════════════════════════

── 1. Shape ──────────────────────────────────────────
Rows    : 57,484,285
Columns : 25
Columns : ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour', 'year', 'CBD_official', 'Division', 'Line', 'Daytime_Routes', 'Structure', 'treated', 'daily_crz_entries', 'quarter_start', 'fare_evasion_pct', 'post', 'did_term', 'time_of_day']

── 2. Year & date coverage ───────────────────────────
year
2024    27012513
2025    30471772
Name: rows, dtype: int64

Date range 2024: 2024-01-01 → 2024-12-31 (expect 2024-01-01 → 2024-12-31)
Date range 2025: 2025-01-01 → 2025-12-31 (expect 2025-01-01 → 2025-12-31)

2026 rows remaining: 0 (expect 0)

── 3. Null check (all columns) ───────────────────────
✅ No nulls anywhere in the dataset

── 4. B

In [18]:
# ── TENTATIVE FILE SIZE ESTIMATE ──────────────────────────────

import sys

# Current memory footprint
mem_mb = df.memory_usage(deep=True).sum() / 1e6

# Estimate CSV size — CSVs on disk are typically 20-40% of in-memory size
# due to no object overhead, just plain text characters
avg_row_size_bytes = sum(
    df[col].astype(str).str.len().mean() + 1  # +1 for comma
    for col in df.columns
)
estimated_csv_bytes = avg_row_size_bytes * len(df)
estimated_csv_gb    = estimated_csv_bytes / 1e9

# Post-compression estimate (gzip typically 70-80% reduction on CSVs)
estimated_gz_gb = estimated_csv_gb * 0.25

print(f"── Tentative Export Size Estimate ───────────────────────")
print(f"Rows                    : {len(df):,}")
print(f"Columns                 : {len(df.columns)}")
print(f"In-memory size          : {mem_mb / 1000:.1f} GB")
print(f"Estimated CSV size      : {estimated_csv_gb:.1f} GB  (uncompressed)")
print(f"Estimated gzip size     : {estimated_gz_gb:.1f} GB  (compressed, ~75% smaller)")
print(f"\nAfter memory reduction  : ~{mem_mb * 0.10 / 1000:.1f} GB in-memory (estimated)")
print(f"After memory reduction CSV: ~{estimated_csv_gb * 0.40:.1f} GB on disk (estimated)")
print(f"\nDisk available          : check Resources panel in Colab (currently 225 GB total)")

── Tentative Export Size Estimate ───────────────────────
Rows                    : 57,484,285
Columns                 : 25
In-memory size          : 43.2 GB
Estimated CSV size      : 11.3 GB  (uncompressed)
Estimated gzip size     : 2.8 GB  (compressed, ~75% smaller)

After memory reduction  : ~4.3 GB in-memory (estimated)
After memory reduction CSV: ~4.5 GB on disk (estimated)

Disk available          : check Resources panel in Colab (currently 225 GB total)


In [19]:
# ══════════════════════════════════════════════════════════════
# MEMORY REDUCTION + DUAL EXPORT (CSV + GZIP)
# ══════════════════════════════════════════════════════════════

# ── MEMORY REDUCTION ──────────────────────────────────────────
print(f"Memory before: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

df['ridership']         = pd.to_numeric(df['ridership'], downcast='integer')
df['transfers']         = pd.to_numeric(df['transfers'], downcast='integer')
df['Hour']              = df['Hour'].astype('int8')
df['treated']           = df['treated'].astype('int8')
df['post']              = df['post'].astype('int8')
df['did_term']          = df['did_term'].astype('int8')
df['year']              = df['year'].astype('int16')
df['daily_crz_entries'] = pd.to_numeric(df['daily_crz_entries'], downcast='float')
df['fare_evasion_pct']  = df['fare_evasion_pct'].astype('float32')
df['latitude']          = df['latitude'].astype('float32')
df['longitude']         = df['longitude'].astype('float32')

for col in ['station_complex', 'borough', 'CRZ_Zone', 'Day_of_Week',
            'fare_class_category', 'Division', 'Line', 'Daytime_Routes',
            'Structure', 'time_of_day']:
    df[col] = df[col].astype('category')

print(f"Memory after : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")

# ── EXPORT 1 — PLAIN CSV ──────────────────────────────────────
path_csv = '/content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_2024_2025.csv'
print("Exporting plain CSV...")
df.to_csv(path_csv, index=False)
size_csv = os.path.getsize(path_csv) / 1e9
print(f"✅ CSV exported")
print(f"   Path : {path_csv}")
print(f"   Size : {size_csv:.2f} GB\n")

# ── EXPORT 2 — GZIP CSV ───────────────────────────────────────
path_gz = '/content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_comp_2024_2025.csv.gz'
print("Exporting gzip CSV...")
df.to_csv(path_gz, index=False, compression='gzip')
size_gz = os.path.getsize(path_gz) / 1e9
print(f"✅ Gzip exported")
print(f"   Path : {path_gz}")
print(f"   Size : {size_gz:.2f} GB\n")

# ── SUMMARY ───────────────────────────────────────────────────
print(f"══ EXPORT COMPLETE ══")
print(f"Rows exported  : {len(df):,}")
print(f"Columns        : {len(df.columns)}")
print(f"Plain CSV      : {size_csv:.2f} GB")
print(f"Gzip CSV       : {size_gz:.2f} GB")
print(f"Space saved    : {size_csv - size_gz:.2f} GB ({((size_csv - size_gz)/size_csv)*100:.0f}% smaller)")
print(f"\nTo reload gzip:")
print(f"  df = pd.read_csv('{path_gz}', compression='gzip')")

Memory before: 43219.0 MB
Memory after : 6081.8 MB

Exporting plain CSV...
✅ CSV exported
   Path : /content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_2024_2025.csv
   Size : 11.31 GB

Exporting gzip CSV...
✅ Gzip exported
   Path : /content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_comp_2024_2025.csv.gz
   Size : 0.31 GB

══ EXPORT COMPLETE ══
Rows exported  : 57,484,285
Columns        : 25
Plain CSV      : 11.31 GB
Gzip CSV       : 0.31 GB
Space saved    : 11.00 GB (97% smaller)

To reload gzip:
  df = pd.read_csv('/content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_comp_2024_2025.csv.gz', compression='gzip')
